# Emergent topology

Reproduces **Fig. 4** and **Figs. S6, S7, S8**.

The hysteresis trajectory ends with the networks moving *away* from a pure distance
kernel while still predicting brain activity. This notebook asks what replaces it, by
tracking the graph structure of the recurrent weights across training.

Analysis is restricted to the **bystander** nodes — those that are neither read-in nor
read-out. Projection constraints impose a tripartite structure on the network by
construction (Fig. S6), so the bystanders are where connectivity is shaped only by task
learning and the spatial embedding.

**Requires:** the trajectory results pickle and the trained models. No fMRI is needed
for Fig. 4; Fig. S7 uses it for the empirical comparison.

Analysis code lives in [`src/trajectory.py`](../src/trajectory.py),
[`src/topology.py`](../src/topology.py) and [`src/dynamics.py`](../src/dynamics.py).

In [ ]:
import os
import warnings

import gymnasium
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.notebook import tqdm

import src.dynamics as dyn
import src.topology as topo
import src.trajectory as traj
import src.utils as utils
from src.config import ensure_dir, get_paths
from src.fmri_io import load_fmri_data

warnings.filterwarnings('ignore')
gymnasium.logger.min_level = gymnasium.logger.ERROR

MODEL_PARAMS = 'model_params_202606d'
SUBSET = 'bystander'      # nodes outside the read-in / read-out projections
SAVE_FIGURES = True

paths = get_paths(MODEL_PARAMS, require='all')
figdir = ensure_dir(paths.figure_dir)

utils.set_font_size(11)
plt.rcParams['svg.fonttype'] = 'none'
sns.set_style('white')
COLORS = utils.get_my_colors(cat_trio=True, as_list=True)


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(os.path.join(figdir, name), dpi=300,
                    bbox_inches='tight', pad_inches=0.01)


print(f'models : {paths.model_dir}')
print(f'figures: {figdir}')

## Inputs

Four of the five metrics are already in the trajectory pickle. Connection length is
computed here, because it needs the parcel coordinates as well as the weights.

In [ ]:
trajectory = traj.load_trajectory(MODEL_PARAMS)

# Bystander nodes exist only where projection constraints do, so the unmasked
# class has no such subset and is not part of this analysis.
results = {r['kernel_label'].strip(): r for r in traj.class_results(trajectory)
           if SUBSET in (r.get('node_subsets') or {})}
BIO, MASKED = 'Masked Eucl.', 'Masked Reg.'

distances = topo.parcel_distances(paths.data_dir)
models = dyn.load_model_table(MODEL_PARAMS, rows=(0, 1, 2))
# Index with iloc rather than iterrows: iterrows does not preserve dtypes,
# and the model builder relies on the stored numpy types.
model_of = {models.iloc[i].class_label: models.iloc[i]
            for i in range(len(models))}
label_of = {BIO: 'bioRNN', MASKED: 'Masked RNN'}

for name, r in results.items():
    print(f"{label_of[name]:<12} {len(r['node_subsets'][SUBSET])} bystander nodes, "
          f"{r['n_runs']} runs, {len(r['sampled_epochs'])} checkpoints")

In [ ]:
METRICS = {
    'degree_topq_mean':    'Mean degree,\ntop 20% of nodes',
    'rich_club_norm_mean': 'Normalized\nrich-club coefficient',
    'clustering_mean':     'Clustering\ncoefficient',
    'participation_mean':  'Participation\ncoefficient',
    'connection_length':   'Connection length,\ntop 20% of edges',
}


def topology_series(result, model_info):
    """Assemble (n_runs, n_epochs) arrays for every metric, plus accuracy."""
    epochs = np.asarray(result['sampled_epochs'], int)
    nodes = np.asarray(result['node_subsets'][SUBSET], int)
    runs = result['runs']
    shape = (len(runs), epochs.size)

    out = {k: np.full(shape, np.nan) for k in METRICS}
    out['accuracy'] = np.full(shape, np.nan)
    out['epochs'] = epochs

    sub = np.ix_(nodes, nodes)
    for ri, run in enumerate(tqdm(runs, desc=label_of[result['kernel_label'].strip()],
                                  leave=False)):
        for ei, epoch in enumerate(epochs):
            record = run['epochs'].get(int(epoch))
            if record is None:
                continue
            out['accuracy'][ri, ei] = record['accuracy']
            stored = (record.get('topology_bystander') or {}).get('metrics') or {}
            for key in METRICS:
                if key in stored:
                    out[key][ri, ei] = stored[key]
            # Connection length needs the weights themselves.
            weights = dyn.load_hidden_weights(model_info, ri, int(epoch), paths.model_dir)
            out['connection_length'][ri, ei] = topo.connection_length_topq(
                weights[sub], distances[sub])
    return out


series = {name: topology_series(results[name], model_of[label_of[name]])
          for name in (BIO, MASKED)}

phases = {}
for name in (BIO, MASKED):
    arrays = traj.epoch_arrays(results[name], fields=('accuracy',), wk_metric='cosine')
    wk_mean, _ = traj.mean_ci(arrays['wk'])
    acc_mean, _ = traj.mean_ci(series[name]['accuracy'])
    # The masked class trains without a kernel, so it has no similarity curve to
    # bend; it inherits the bioRNN onset so the two are compared over one window.
    phases[name] = (traj.phase_onsets(wk_mean, acc_mean)
                    if np.isfinite(wk_mean).any() else phases[BIO])

onset = phases[BIO]['phase_iii']
print(f"\nphase III begins at epoch {series[BIO]['epochs'][onset]:,}")

## Fig. 4 — connectome-like topology emerges during training

Each panel tracks one graph property of the bystander subnetwork across the third
training phase, with task accuracy overlaid. Together they describe a network becoming
both more segregated (clustering) and more integrated (participation, long-range
connections), organised around a densely interconnected hub core (degree, rich-club).

The final panel shows how fast each property is changing: reorganisation is concentrated
early in the phase and settles as accuracy approaches its ceiling — topology leading
behaviour rather than following it.

In [ ]:
def normalize_unit(y):
    """Min-max a curve onto [0, 1] so metrics with different units compare."""
    lo, hi = np.nanmin(y), np.nanmax(y)
    return (y - lo) / (hi - lo) if hi > lo else np.zeros_like(y)


def topology_figure(name, filename):
    data = series[name]
    start = phases[name]['phase_iii']
    epochs = data['epochs'][start:]
    acc_mean, _ = traj.mean_ci(data['accuracy'][:, start:])

    fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
    for ax, (key, title), color in zip(axes.ravel(), METRICS.items(), COLORS * 2):
        mean, ci = traj.mean_ci(data[key][:, start:])
        ax.plot(epochs, mean, color=color, lw=2)
        ax.fill_between(epochs, mean - ci, mean + ci, color=color, alpha=0.2, lw=0)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Training epoch')
        twin = ax.twinx()                      # accuracy on every panel, for timing
        twin.plot(epochs, acc_mean * 100, color='k', lw=1, ls='--', alpha=0.7)
        twin.set_ylim(0, 100)
        twin.set_ylabel('Accuracy (%)', fontsize=9)

    # Rate of change, on a common scale.
    ax = axes.ravel()[-1]
    for (key, title), color in zip(METRICS.items(), COLORS * 2):
        mean, _ = traj.mean_ci(data[key][:, start:])
        rate = np.gradient(traj.smooth(normalize_unit(mean)), epochs) * 1000
        ax.plot(epochs, rate, color=color, lw=1.5, label=title.replace('\n', ' '))
    ax.axhline(0, color='0.5', lw=1)
    ax.set_title('Rate of change\n(normalized, per 1000 epochs)', fontsize=10)
    ax.set_xlabel('Training epoch')
    ax.legend(frameon=False, fontsize=7, loc='upper right')

    fig.suptitle(f'{label_of[name]} — bystander subnetwork', fontsize=12)
    sns.despine(fig=fig, right=False, top=True)
    fig.tight_layout()
    save(fig, filename)
    plt.show()


topology_figure(BIO, 'fig4_topology_biornn.svg')

## Fig. S8 — topology in the Masked RNNs

The masked class develops some of the same properties, but without the accompanying
growth in long-range connections — it was never exposed to the geometry that would make
distance costly, so there is nothing for it to trade against.

In [ ]:
topology_figure(MASKED, 'figS8_topology_masked.svg')

## Fig. S6 — recurrent weights across training

Snapshots of the full hidden-to-hidden weight matrix. The tripartite block structure
imposed by the read-in and read-out projections is visible throughout, which is why the
topology analysis above is restricted to the bystander block. The rightmost panel shows
the Euclidean kernel the networks were trained against.

In [ ]:
from src.fmri_io import load_kernels

SNAPSHOT_FRACTIONS = (0.0, 0.05, 0.25, 0.5, 1.0)
epochs_all = series[BIO]['epochs']
snapshot_epochs = [int(epochs_all[int(f * (epochs_all.size - 1))])
                   for f in SNAPSHOT_FRACTIONS]

kernel = load_kernels(paths.data_dir)['euclidean']
bio_model = model_of['bioRNN']

fig, axes = plt.subplots(1, len(snapshot_epochs) + 1,
                         figsize=(2.1 * (len(snapshot_epochs) + 1), 2.3))
for ax, epoch in zip(axes, snapshot_epochs):
    weights = dyn.load_hidden_weights(bio_model, 0, epoch, paths.model_dir)
    lim = np.percentile(np.abs(weights), 99)
    ax.imshow(weights, cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_title(f'epoch {epoch:,}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

axes[-1].imshow(kernel, cmap='viridis')
axes[-1].set_title('Euclidean kernel', fontsize=9)
axes[-1].set_xticks([]); axes[-1].set_yticks([])
fig.tight_layout()
save(fig, 'figS6_weight_snapshots.svg')
plt.show()

## Fig. S7 — functional connectivity of the bystander subnetwork

Correlating the networks' own hidden activity gives a model-side counterpart to an
empirical FC matrix. Modular structure appears over training and comes to resemble the
empirical pattern (rightmost), in nodes shaped only by task learning and the embedding.

Values are Fisher z-transformed correlations; colour limits are robust percentiles, since
a few strongly coupled pairs would otherwise dominate the scale.

In [ ]:
nodes = np.asarray(results[BIO]['node_subsets'][SUBSET], int)

fig, axes = plt.subplots(1, len(snapshot_epochs) + 1,
                         figsize=(2.1 * (len(snapshot_epochs) + 1), 2.3))
for ax, epoch in zip(axes, snapshot_epochs):
    evaluated = dyn.evaluate_run(bio_model, 0, epoch, paths.model_dir,
                                 rest_nsteps=1200, n_pc=5, return_hidden=True)
    fc = dyn.model_functional_connectivity(evaluated['hidden_task'], nodes)
    lim = np.nanpercentile(np.abs(fc), 95)
    ax.imshow(fc, cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_title(f'epoch {epoch:,}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

# Empirical comparison: group-average task fMRI FC over the same nodes.
fmri_task, _, _, _ = load_fmri_data(paths.data_dir, paths.fmri_dir, 100,
                                    hidden_size=100, verbose=False)
empirical = np.nanmean(
    np.stack([utils.compute_fc(fmri_task[:, nodes, s])
              for s in range(fmri_task.shape[2])]), axis=0)
lim = np.nanpercentile(np.abs(empirical), 95)
axes[-1].imshow(empirical, cmap='RdBu_r', vmin=-lim, vmax=lim)
axes[-1].set_title('Empirical (task fMRI)', fontsize=9)
axes[-1].set_xticks([]); axes[-1].set_yticks([])
fig.tight_layout()
save(fig, 'figS7_model_functional_connectivity.svg')
plt.show()

## Reported values

In [ ]:
print(f'Bystander subnetwork, across phase III '
      f'(from epoch {series[BIO]["epochs"][onset]:,})\n')
header = f'{"metric":<34}{"bioRNN start → end":>26}{"Masked start → end":>26}'
print(header + '\n' + '-' * len(header))
for key, title in METRICS.items():
    cells = []
    for name in (BIO, MASKED):
        start = phases[name]['phase_iii']
        mean, _ = traj.mean_ci(series[name][key][:, start:])
        cells.append(f'{mean[0]:10.3f} → {mean[-1]:<10.3f}')
    print(f'{title.replace(chr(10), " "):<34}{cells[0]:>26}{cells[1]:>26}')

for name in (BIO, MASKED):
    acc, _ = traj.mean_ci(series[name]['accuracy'])
    print(f'\n{label_of[name]}: accuracy {acc[0]*100:.0f}% → {acc[-1]*100:.0f}% '
          f'over {series[name]["accuracy"].shape[0]} runs')